In [ ]:
!pip install h5py

import os
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Subset
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

In [ ]:
for lib in ["os", "h5py", "torch", "pandas", "numpy", "sklearn", "tqdm", "matplotlib"]:
    try:
        __import__(lib)
        print(f"{lib} OK")
    except ImportError:
        print(f"{lib} not installed")


In [ ]:
batch_size = 128
num_epochs = 50
lr = 5e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"I use: {device}")
class VideoClipDataset(Dataset):
    """
    Dataset PyTorch for features extracted by V-JEPA2 model.
    It supports multiple HDF5 files.
    """
    def __init__(self, h5_paths):
        if isinstance(h5_paths, str):
            h5_paths = [h5_paths]

        self.data = []
        self.labels = []
        self.clip_ids = []
        self.video_ids = []

        for h5_path in h5_paths:
            print(f"Load features from: {h5_path}")
            with h5py.File(h5_path, "r") as f:
                for clip_name in f.keys():
                    grp = f[clip_name]
                    feat = grp["features"][()]          # shape (32, 1408)
                    label = int(grp["label"][()])       # 0 or 1
                    video_id = grp.attrs["video_id"]

                    self.data.append(feat.astype("float32"))
                    self.labels.append(label)
                    self.clip_ids.append(clip_name)
                    self.video_ids.append(video_id)

        
        self.data = torch.from_numpy(np.array(self.data, dtype=np.float32))   
        self.labels = torch.tensor(self.labels, dtype=torch.float32)          

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx], self.video_ids[idx]


In [ ]:
# Loading of multiple files
h5_files = [
    "/kaggle/input/features/all_clips_features_1_1004_merged.h5",
    "/kaggle/input/features/clips_features_1005_2004.h5",
    "/kaggle/input/features/clips_features_2005_2804.h5",
    "/kaggle/input/features/clips_features_2805_3319.h5",
    "/kaggle/input/features/clips_features_3320_3954.h5"
]

dataset = VideoClipDataset(h5_files)
print("Total number of clips:", len(dataset))


In [ ]:
labels = np.array([y for _, y, _ in dataset]) 
num_pos = np.sum(labels == 1)
num_neg = np.sum(labels == 0)
total = len(labels)

print(f"Total clips: {total}")
print(f"Positives (fights): {num_pos} → {100*num_pos/total:.2f}%")
print(f"Negatives (normal): {num_neg} → {100*num_neg/total:.2f}%")

In [ ]:
print("Split the dataset video-level (GroupShuffleSplit)...")

video_ids = np.array(dataset.video_ids)
indices = np.arange(len(dataset))

# Split: 80% train, 20% val
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(indices, groups=video_ids))

# Subset PyTorch creation
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

print(f"Train clips: {len(train_dataset)}, Val clips: {len(val_dataset)}")

# Balancing check
def cls_stats(subset):
    labels = np.array([dataset[i][1].item() for i in subset.indices])
    num_pos = int(labels.sum())
    num_tot = len(labels)
    return num_tot, num_pos, 100 * num_pos / num_tot

print("Train:", cls_stats(train_dataset))
print("Val:", cls_stats(val_dataset))

# Change to use WeightedRandomSampler
train_labels = np.array([dataset[i][1].item() for i in train_dataset.indices])

class_sample_count = np.array([(train_labels == t).sum() for t in [0, 1]], dtype=np.float64)
print(f"Distribution in train set: {class_sample_count.tolist()}")

# Class frequency inverse
weight_per_class = 1.0 / class_sample_count

# Weight for each sample based on its label
samples_weight = np.array([weight_per_class[int(y)] for y in train_labels])
samples_weight = torch.from_numpy(samples_weight).double()

# Balanced Sampler 
sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True  # Note: allows resampling of minority class samples
)

# DataLoader with sampler
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=4,
    pin_memory=True
)

# Validation loader (no balancing)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# =========================
# MODEL 1D-CNN 
# =========================
class FightCNN1D(nn.Module):
    """
    Use combined global pooling (Max + Avg) to capture more information.
    """
    def __init__(self, input_dim=1408):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 512, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(512)
        self.conv2 = nn.Conv1d(512, 512, kernel_size=3, stride=2, padding=1)  # 32 -> 16
        self.bn2 = nn.BatchNorm1d(512)
        self.conv3 = nn.Conv1d(512, 256, kernel_size=3, stride=2, padding=1)  # 16 -> 8
        
         
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        
        
        self.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.4), 
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 1408, 32)
        x = nn.functional.relu(self.bn1(self.conv1(x)))
        x = nn.functional.relu(self.bn2(self.conv2(x)))
        x = nn.functional.relu(self.conv3(x))
        
        
        x_max = self.global_max_pool(x).squeeze(-1)
        x_avg = self.global_avg_pool(x).squeeze(-1)
        x = torch.cat((x_max, x_avg), dim=1)  # (B, 512)
        
        # Final classifier
        return self.fc(x).squeeze(-1)

class FightCNN1D_Attention(nn.Module):
    """
    1D CNN with a lightweight temporal attention block.
    The CNN extracts features, then an attention layer learns to weight the most relevant frames.
    """
    def __init__(self, input_dim=1408):
        super().__init__()
        
        # --- Convolutional Blocks ---
        self.conv1 = nn.Conv1d(input_dim, 512, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(512)
        self.conv2 = nn.Conv1d(512, 512, kernel_size=3, stride=2, padding=1)  # 32 → 16
        self.bn2 = nn.BatchNorm1d(512)
        self.conv3 = nn.Conv1d(512, 256, kernel_size=3, stride=2, padding=1)  # 16 → 8
        self.bn3 = nn.BatchNorm1d(256)

        # --- Temporal Attention Block ---
        # Produces a weight (between 0 and 1) for each timestep in the sequence (8 in our case)
        self.attention = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=1),  
            nn.Tanh(),
            nn.Conv1d(128, 1, kernel_size=1),   
            nn.Softmax(dim=-1)                  
        )

        # --- Final classifier ---
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        # Input: (B, 32, 1408)
        x = x.transpose(1, 2)  # (B, 1408, 32)
        x = F.relu(self.bn1(self.conv1(x)))  # (B, 512, 32)
        x = F.relu(self.bn2(self.conv2(x)))  # (B, 512, 16)
        x = F.relu(self.bn3(self.conv3(x)))  # (B, 256, 8)

        # Computes the attention map
        attn_weights = self.attention(x)  # (B, 1, 8)
        x = torch.sum(x * attn_weights, dim=-1)  # (B, 256) → temporal weighted sum

        # Final classification
        logits = self.fc(x).squeeze(-1)
        return logits

print("Model loading ...")
#model = FightCNN1D().to(device) # Choose this for the first classifier
model = FightCNN1D_Attention().to(device) # Choose this for the second classifier

# =========================
# LOSS & OPTIMIZER
# =========================
criterion = nn.BCEWithLogitsLoss()  
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)


In [ ]:
# Execute this cell if you want to use a temporal transformer

print("Split the dataset video-level (GroupShuffleSplit)...")

video_ids = np.array(dataset.video_ids)
indices = np.arange(len(dataset))

# Split: 80% train, 20% val
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(indices, groups=video_ids))

# Subset PyTorch creation
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

print(f"Train clips: {len(train_dataset)}, Val clips: {len(val_dataset)}")

# Balancing check
def cls_stats(subset):
    labels = np.array([dataset[i][1].item() for i in subset.indices])
    num_pos = int(labels.sum())
    num_tot = len(labels)
    return num_tot, num_pos, 100 * num_pos / num_tot

print("Train:", cls_stats(train_dataset))
print("Val:", cls_stats(val_dataset))

# Change to use WeightedRandomSampler
train_labels = np.array([dataset[i][1].item() for i in train_dataset.indices])


class_sample_count = np.array([(train_labels == t).sum() for t in [0, 1]], dtype=np.float64)
print(f"Distribution in the train set: {class_sample_count.tolist()}")

# Class frequency inverse
weight_per_class = 1.0 / class_sample_count

# Weight for each sample based on its label
samples_weight = np.array([weight_per_class[int(y)] for y in train_labels])
samples_weight = torch.from_numpy(samples_weight).double()

# Sampler balanced
sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True  # Note: allows resampling of minority class samples
)

# DataLoader with sampler
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=4,
    pin_memory=True
)

# Validation loader (no balancing)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

class PositionalEncoding(nn.Module):
    # Implements standard sinusoidal positional encoding.
    def __init__(self, d_model, max_len=128):
        super().__init__()
        # Create a (max_len, d_model) tensor filled with zeros
        pe = torch.zeros(max_len, d_model)
        # Create position indices (0, 1, ..., max_len-1)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # Compute the sinusoidal frequencies for even and odd dimensions
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        # Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        # Register pe as a buffer (not a parameter) with shape (1, max_len, d_model)
        self.register_buffer('pe', pe.unsqueeze(0))  
    def forward(self, x):
        # x: (B, T, D)
        # Add positional encoding to input embeddings
        return x + self.pe[:, :x.size(1), :]

class TemporalTransformerClassifier(nn.Module):
    #1D Transformer-based classifier with attention pooling for temporal sequences.
    def __init__(self, input_dim=1408, d_model=512, nhead=8, num_layers=3, dim_feedforward=1024, dropout=0.2):
        super().__init__()
        # Project input features to model dimension
        self.proj = nn.Linear(input_dim, d_model)
         # Positional encoding for temporal order
        self.pos = PositionalEncoding(d_model, max_len=64)
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu')
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Attention pooling layer: computes per-timestep attention logits
        self.attn_pool = nn.Sequential(
            nn.Linear(d_model, d_model//4),
            nn.ReLU(),
            nn.Linear(d_model//4, 1)
        )

        # Classification head
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(d_model//2, 1)
        )
    def forward(self, x):
        # x: (B, T, input_dim)
        x = self.proj(x)                # Project to (B, T, d_model)
        x = self.pos(x)                 # Add positional encoding
        x = x.transpose(0,1)            # Transformer expects (T, B, D)
        x = self.encoder(x)             # Encode temporal features (T, B, D)
        x = x.transpose(0,1)            # Back to (B, T, D)
        
        # Attention pooling
        attn_logits = self.attn_pool(x).squeeze(-1)   # (B, T)
        attn_w = torch.softmax(attn_logits, dim=1).unsqueeze(-1)  # (B, T, 1)
        pooled = (x * attn_w).sum(dim=1)  # Weighted sum over time (B, D)

        # Classification head
        out = self.head(pooled).squeeze(-1)  # Output logit
        return out  # Logits (not passed through sigmoid)

print("Model loading ...")
model = TemporalTransformerClassifier().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def find_best_threshold(y_true, y_prob):
    """
    Search for the threshold between 0.1 and 0.9 that maximizes the F1 score
    on the validation set.

    Args:
        y_true (array-like): True binary labels (0 or 1).
        y_prob (array-like): Predicted probabilities.

    Returns:
        best_t (float): Threshold that gives the highest F1 score.
        best_f1 (float): The corresponding F1 score.
    """
    thresholds = np.linspace(0.1, 0.9, 81)
    best_t, best_f1 = 0.5, 0
    for t in thresholds:
        preds = (y_prob > t).astype(int)
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t, best_f1


In [ ]:
# =========================
# FUNCTION TO COMPUTE METRICS
# =========================

def compute_metrics(labels, preds_prob, video_ids=None):
    """
    Compute standard metrics and AUCA/APA if video_ids are provided.

    Args:
        labels (array-like, shape (N,)): True labels for N clips (0 or 1).
        preds_prob (array-like, shape (N,)): Predicted probabilities for N clips.
        video_ids (array-like, optional): Video IDs corresponding to each clip.

    Returns:
        dict: Dictionary containing computed metrics.
    """
    # Find the threshold that maximizes F1 score on the validation set
    best_threshold, best_f1 = find_best_threshold(labels, preds_prob)
    print(f"Soglia ottimale = {best_threshold:.3f}, F1 = {best_f1:.3f}")

    # Convert probabilities to binary predictions using the optimal threshold
    preds_bin = (preds_prob > best_threshold).astype(int) 
    
    metrics = {}
    metrics['ROC-AUC'] = roc_auc_score(labels, preds_prob)
    metrics['AP'] = average_precision_score(labels, preds_prob)
    metrics['Precision'] = precision_score(labels, preds_bin)
    metrics['Recall'] = recall_score(labels, preds_bin)
    metrics['F1'] = f1_score(labels, preds_bin)
    
    if video_ids is not None:
        # Create a DataFrame containing clip-level information
        # Each row represents a clip with its video_id, true label, and predicted probability
        # Example structure:
        # df =
        #    video_id   label   pred
        #    0  video_001     0   0.12
        #    1  video_001     1   0.83
        #    2  video_002     0   0.10
        #    3  video_002     0   0.05
        #    4  video_003     1   0.90

        df = pd.DataFrame({
            "video_id": video_ids,
            "label": labels,
            "pred": preds_prob
        })
        
        # Select videos with at least one positive clip (“fight”)
        # Group clips by video and take max label per video
        # If any clip is positive, max label = 1 → video is abnormal
        abnormal_videos = df.groupby("video_id")["label"].max() 
        abnormal_videos = abnormal_videos[abnormal_videos == 1].index # list of video IDs with at least one positive clip
        
        aucs, aps = [], []
        for vid in abnormal_videos:
            df_vid = df[df["video_id"] == vid]
            # Skip videos where all clips have the same label (all 1) 
            # because we cannot compute meaningful AUCA/APA
            if len(df_vid["label"].unique()) == 1:
                continue 
            # Compute AUCA and APA within the video
            # Measures how well the model distinguishes fight vs normal clips in this video
            aucs.append(roc_auc_score(df_vid["label"], df_vid["pred"]))
            aps.append(average_precision_score(df_vid["label"], df_vid["pred"]))
        
        metrics['AUCA'] = np.mean(aucs) if aucs else np.nan
        metrics['APA'] = np.mean(aps) if aps else np.nan

    # Return a dictionary with all computed metrics
    # Example:
    # {
    # 'ROC-AUC': 0.92,
    # 'AP': 0.88,
    # 'Precision': 0.80,
    # 'Recall': 0.75,
    # 'F1': 0.77,
    # 'AUCA': 0.85,
    # 'APA': 0.83
    # }

    return metrics

In [ ]:
# =========================
# TRAINING LOOP
# =========================

# Lists to store training metrics over epochs
train_losses_list = []
f1_list = []
roc_auc_list = []
auca_list = []
apa_list = []

best_f1 = 0.0
patience = 10          # Number of consecutive epochs to tolerate without improvement
min_delta = 0.002      # Minimum F1 improvement to be considered a “real” improvement
epochs_no_improve = 0  # Counter for epochs without significant improvement

for epoch in range(num_epochs):
    model.train()
    train_losses = []

     # -------------------------
    # TRAINING
    # -------------------------
    for X, y, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Train"):
        X, y = X.to(device), y.to(device).float()
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    avg_train_loss = sum(train_losses)/len(train_losses)
    train_losses_list.append(avg_train_loss)

    # =========================
    # VALIDATION
    # =========================
    model.eval()
    val_labels = []
    val_preds_prob = []
    val_video_ids = []


    with torch.no_grad():
        for X_val, y_val, vid_ids in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            outputs = model(X_val)

            # Save predictions and true labels for metric calculation
            probs = torch.sigmoid(outputs)  # Convert logits to probabilities
            val_preds_prob.extend(probs.cpu().numpy())
            val_labels.extend(y_val.cpu().numpy())
            val_video_ids.extend(vid_ids)
    
    val_labels = np.array(val_labels)
    val_preds_prob = np.array(val_preds_prob)

    # Compute all metrics using the previously defined function
    metrics = compute_metrics(val_labels, val_preds_prob, val_video_ids)

    # Save metrics for plotting
    f1_list.append(metrics['F1'])
    roc_auc_list.append(metrics['ROC-AUC'])
    auca_list.append(metrics['AUCA'])
    apa_list.append(metrics['APA'])

    # Print metrics for the current epoch
    print(
    f"Epoch {epoch+1}: TrainLoss={avg_train_loss:.4f}, "
    f"ROC-AUC={metrics['ROC-AUC']:.4f}, AP={metrics['AP']:.4f}, "
    f"Precision={metrics['Precision']:.4f}, Recall={metrics['Recall']:.4f}, F1={metrics['F1']:.4f}, "
    f"AUCA={metrics['AUCA']:.4f}, APA={metrics['APA']:.4f}"
    )

    # Save best model based on F1
    if metrics['F1'] > best_f1 + min_delta:
        best_f1 = metrics['F1']
        torch.save(model.state_dict(), "best_fight_cnn1d.pth")
        print(f"New best model saved (F1={best_f1:.4f})!")
        epochs_no_improve = 0  # reset counter
    else:
        epochs_no_improve += 1
        print(f" No real F1 improvement for {epochs_no_improve} consecutive epochs.")

    # Early stopping check
    if epochs_no_improve >= patience:
        print(f"\n Early stopping triggered: no improvement >{min_delta} for {patience} epochs.")


print(f"\nTraining complete! Best F1 achieved: {best_f1:.4f}")

# =========================
# METRICS PLOTTING
# =========================
epochs_range = range(1, len(f1_list)+1)
plt.figure(figsize=(12, 6))
plt.plot(epochs_range, f1_list, label='F1')
plt.plot(epochs_range, roc_auc_list, label='ROC-AUC')
plt.plot(epochs_range, auca_list, label='AUCA')
plt.plot(epochs_range, apa_list, label='APA')
plt.xlabel("Epoch")
plt.ylabel("Metric value")
plt.title("Training Metrics Over Epochs")
plt.legend()
plt.grid(True)
plt.show()



In [ ]:
# =========================
# FINAL TEST (INFERENCE)
# =========================

# Load the test dataset
test_dataset = VideoClipDataset("/kaggle/input/features/clips_features_testset_review.h5")
print(f"Number of test clips:  {len(test_dataset)}")

# Create DataLoader for test set
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
print("\nLoading the best saved model for testing...")

# Load the best model saved during training
model.load_state_dict(torch.load("best_fight_cnn1d.pth", map_location=device))
model.eval()

# Lists to store predictions and labels
test_labels = []
test_preds_prob = []
test_video_ids = []

with torch.no_grad():
    for X_test, y_test, vid_ids in tqdm(test_loader, desc="Testing"):
        X_test, y_test = X_test.to(device), y_test.to(device)
        outputs = model(X_test)
        probs = torch.sigmoid(outputs)  # Convert logits to probabilities

        # Store predictions and true labels
        test_preds_prob.extend(probs.cpu().numpy())
        test_labels.extend(y_test.cpu().numpy())
        test_video_ids.extend(vid_ids)

# Convert lists to numpy arrays
test_labels = np.array(test_labels)
test_preds_prob = np.array(test_preds_prob)

# Compute global and per-video metrics
test_metrics = compute_metrics(test_labels, test_preds_prob, test_video_ids)

# Also find the optimal F1 threshold on the test set (for analysis)
best_thr, best_f1_thr = find_best_threshold(test_labels, test_preds_prob)

# Print final results
print("\n==============================")
print(" FINAL TEST SET RESULTS")
print("==============================")
print(f"Soglia ottimale per F1: {best_thr:.2f} (F1={best_f1_thr:.4f})")
print(f"ROC-AUC   = {test_metrics['ROC-AUC']:.4f}")
print(f"AP        = {test_metrics['AP']:.4f}")
print(f"Precision = {test_metrics['Precision']:.4f}")
print(f"Recall    = {test_metrics['Recall']:.4f}")
print(f"F1        = {test_metrics['F1']:.4f}")
print(f"AUCA      = {test_metrics['AUCA']:.4f}")
print(f"APA       = {test_metrics['APA']:.4f}")
print("==============================")
